In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_dataset=pd.read_csv('/content/drive/MyDrive/Titanic-dataset/train.csv')
test_dataset=pd.read_csv('/content/drive/MyDrive/Titanic-dataset/test.csv')

In [4]:
titanic_train=train_dataset.copy()
titanic_test=test_dataset.copy()

In [5]:
og_train=train_dataset.copy()
og_test=test_dataset.copy()

In [6]:
train_dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
train_dataset.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [8]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [9]:
train_dataset['Embarked'].unique()

array(['S', 'C', 'Q', nan], dtype=object)

In [10]:
import re

In [11]:
data=train_dataset['Cabin'].str.extract(r'([a-zA-Z])',expand=False)

In [12]:
data.value_counts()

,count
Cabin,
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [13]:
train_dataset[train_dataset.loc[:,'Cabin']=='T']

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
339,340,0,1,"Blackwell, Mr. Stephen Weart",male,45.0,0,0,113784,35.5,T,S


In [14]:
train_dataset['Ticket']

,Ticket
0,A/5 21171
1,PC 17599
2,STON/O2. 3101282
3,113803
4,373450
...,...
886,211536
887,112053
888,W./C. 6607
889,111369


In [15]:
train_dataset['Ticket'].str.extract(r'(?:\d+)*(\w+[\.]?[\/]?\w+[\.]?)\s*(\d+)*',expand=False)

,0,1
0,A/5,21171
1,PC,17599
2,STON/O2.,3101282
3,03,NaN
4,50,NaN
...,...,...
886,36,NaN
887,53,NaN
888,W./C.,6607
889,69,NaN


In [16]:
print(train_dataset['Name'].to_string())

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
5                                       Moran, Mr. James
6                                McCarthy, Mr. Timothy J
7                         Palsson, Master. Gosta Leonard
8      Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)
9                    Nasser, Mrs. Nicholas (Adele Achem)
10                       Sandstrom, Miss. Marguerite Rut
11                              Bonnell, Miss. Elizabeth
12                        Saundercock, Mr. William Henry
13                           Andersson, Mr. Anders Johan
14                  Vestrom, Miss. Hulda Amanda Adolfina
15                       Hewlett, Mrs. (Mary D Kingcome)
16                                  Rice, Master. Eugene
17                          Wil

In [17]:
train_dataset['Title']=train_dataset['Name'].str.extract(r'\,\s+(\w+)')

In [18]:
train_dataset['Title'].unique()

array(['Mr', 'Mrs', 'Miss', 'Master', 'Don', 'Rev', 'Dr', 'Mme', 'Ms',
       'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'the', 'Jonkheer'],
      dtype=object)

In [19]:
train_dataset.loc[train_dataset['Title'].isin(['Mlle','Ms']),'Title']='Miss'

In [20]:
train_dataset.loc[train_dataset['Title'].isin(['Major','Col','Capt']),'Title']='Army'

In [21]:
train_dataset.loc[train_dataset['Title'].isin(['Sir','Don','Jonkheer','Lady','the','Mme','Army','Rev','Dr']),'Title']='Rare'

In [22]:
train_dataset['Title'].value_counts()

,count
Title,
Mr,517
Miss,185
Mrs,125
Master,40
Rare,24


In [23]:
train_dataset['Name'].str.extract(r'(\w+)')

,0
0,Braund
1,Cumings
2,Heikkinen
3,Futrelle
4,Allen
...,...
886,Montvila
887,Graham
888,Johnston
889,Behr


In [24]:
train_dataset.drop(columns=['Name'],inplace=True)

In [25]:
train_dataset.loc[train_dataset['Parch'].isin([0,1,2])]['Survived'].value_counts()/len(train_dataset.loc[train_dataset['Parch'].isin([0,1,2])])
#after 0 rate to survive decreases , if people have their family members on board they will probably try to save them

,count
Survived,
0,0.614155
1,0.385845


In [26]:
train_dataset['Boogie']=train_dataset['Cabin'].str.extract(r'([a-zA-Z])')

In [27]:
train_dataset.groupby('Boogie')['Survived'].value_counts(normalize=True)

Boogie  Survived
A       0           0.533333
        1           0.466667
B       1           0.744681
        0           0.255319
C       1           0.593220
        0           0.406780
D       1           0.757576
        0           0.242424
E       1           0.750000
        0           0.250000
F       1           0.615385
        0           0.384615
G       0           0.500000
        1           0.500000
T       0           1.000000
Name: proportion, dtype: float64

In [28]:
pd.crosstab(
    train_dataset['Boogie'],
    train_dataset['Survived'],
    normalize='index'
) * 100

Survived,0,1
Boogie,,
A,53.333333,46.666667
B,25.531915,74.468085
C,40.677966,59.322034
D,24.242424,75.757576
E,25.000000,75.000000
F,38.461538,61.538462
G,50.000000,50.000000
T,100.000000,0.000000


In [29]:
train_dataset.loc[train_dataset['Boogie'].isin(['T']),'Boogie']=train_dataset['Boogie'].value_counts().index[0]
#replaced 'T' with most frequent value

In [30]:
from google.colab import files
train_dataset['Ticket'].to_csv('ticket.csv',index=False)

files.download('ticket.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# ticket_number=train_dataset['Ticket'].str.extract(r'.*?(\d+).*?(\d+)').astype(float)
# train_dataset['Ticket_number']= pd.concat([
#     ticket_number.loc[ticket_number[0] > ticket_number[1], 0],
#     ticket_number.loc[ticket_number[1] > ticket_number[0], 1]
# ]).sort_index() #first condition selects where right ticket is at 1st column and second condition where is it at 2nd column.
#                 #now concatenates put them over one another
#                 #indexing value is as it is and through sort_index we give it a rightful order
# better solution came yo!

In [32]:
train_dataset['Ticket_number']=train_dataset['Ticket'].str.extract(r'(\d+)$')
#$ means this pattern of digits must be at end
# if ^(\d)+ means this digit should occur at beginning.

In [33]:
train_dataset.groupby('Ticket_number')['PassengerId'].value_counts()

Ticket_number  PassengerId
10482          284            1
110152         258            1
               505            1
               760            1
110413         263            1
                             ..
8475           653            1
851            154            1
9234           301            1
9549           11             1
               395            1
Name: count, Length: 887, dtype: int64

In [34]:
train_dataset.groupby('Ticket_number')['Survived'].agg(['count', 'mean']).sort_values(by='count',ascending=False)

,count,mean
Ticket_number,,
347082,7,0.000000
1601,7,0.714286
2343,7,0.000000
347088,6,0.000000
3101295,6,0.000000
...,...,...
111369,1,1.000000
111320,1,0.000000
111240,1,0.000000


In [35]:
train_dataset.groupby('Ticket_number')['PassengerId'].agg(['count']).sort_values(by='count',ascending=False)

,count
Ticket_number,
347082,7
1601,7
2343,7
347088,6
3101295,6
...,...
111369,1
111320,1
111240,1


In [36]:
train_dataset['Ticket'].isnull()

,Ticket
0,False
1,False
2,False
3,False
4,False
...,...
886,False
887,False
888,False
889,False


In [37]:
train_dataset['Fare'].std()

49.693428597180905

In [38]:
train_dataset['Fare'].mean()

np.float64(32.204207968574636)

In [39]:
train_dataset['Fare'].quantile(0.75)-train_dataset['Fare'].quantile(0.25)

np.float64(23.0896)

In [40]:
train_dataset['Fare'].quantile(0.25)

np.float64(7.9104)

In [41]:
train_dataset['Fare'].quantile(0.75)

np.float64(31.0)

In [42]:
train_dataset['Fare'].quantile(0.90)-train_dataset['Fare'].quantile(0.75)

np.float64(46.958299999999994)

In [43]:
from statsmodels import robust

In [44]:
robust.scale.mad(train_dataset['Fare'])

np.float64(10.236182237006377)

In [45]:
train_dataset['Fare'].std()

49.693428597180905

In [46]:
og_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [47]:
from sklearn.base import TransformerMixin, BaseEstimator

In [48]:
class TitanicEngineer(TransformerMixin, BaseEstimator):
  def __init__(self,X):
    self.X=X


  def fit(self,X):
    return self

  def transform(self,X):
    X=X.copy()
    X['Title']=X['Name'].str.extract(r'\,\s+(\w+)')
    X.loc[X['Title'].isin(['Mlle','Ms']),'Title']='Miss'
    X.loc[X['Title'].isin(['Major','Col','Capt']),'Title']='Army'
    X.loc[X['Title'].isin(['Sir','Don','Jonkheer','Lady','the','Mme','Army','Rev','Dr']),'Title']='Rare'
    X['Cabin']=X['Cabin'].str.extract(r'([a-zA-Z])',expand=False)
    X.loc[X['Cabin'].notna() & ~X['Cabin'].isin(['C', 'E', 'G', 'D', 'A', 'B', 'F']),'Cabin']=X['Cabin'].value_counts().index[0]
    return X

In [49]:
titan=TitanicEngineer(og_train)
titan.fit_transform(og_train)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,C,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C,S,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Rare
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B,S,Miss
888,889,0,3,"Johnston, Miss. Catherine Helen 'Carrie'",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Miss
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C,C,Mr


# **Problems in TitanicEngineer**
1 -> there is no inplace transformation i would have to store the result of titan.fit_transform(og_train)<br>
2 -> earlier in initialization i returned only self and nothing was passed when titanic called  TitanicEngineer() with ot without creating an object i got this error-
TypeError: __init__() should return None, not 'TitanicEngineer'<br>
3 -> how should be the structure and how should fit be and transform and when class is created, as an constructor what function should we pass it?
<br>
# **Answer**<br>
1-> in transform remove x.copy for inplace transform<br>
2-> init is not supposed to return and dont use dataset, it is supposed to intialize parameters only<br>
3-> see next custom

In [50]:
class TitanicDP(TransformerMixin,BaseEstimator):
  def __init__(self):
    pass

  def fit(self,X):
    self.most_occupied_cabin=X['Cabin'].str.extract(r'([a-zA-Z])',expand=False).value_counts().index[0]
    return self

  def transform(self,X):

    X=X.copy()
    X['Title']=X['Name'].str.extract(r'\,\s+(\w+)')
    X.loc[X['Title'].isin(['Mlle','Ms']),'Title']='Miss'
    X.loc[X['Title'].isin(['Major','Col','Capt']),'Title']='Army'
    X.loc[X['Title'].isin(['Sir','Don','Jonkheer','Lady','the','Mme','Army','Rev','Dr']),'Title']='Rare'
    X['Cabin']=X['Cabin'].str.extract(r'([a-zA-Z])',expand=False)
    X.loc[X['Cabin'].notna() & ~X['Cabin'].isin(['C', 'E', 'G', 'D', 'A', 'B', 'F']),'Cabin']=self.most_occupied_cabin
    return X

  #generally both x and y are passed but i have not separated by my dataset till yet so fit only have x

In [73]:
class TitanicDP(TransformerMixin,BaseEstimator):
  def __init__(self):
    pass

  def fit(self,X,y=None):
    self.most_occupied_cabin_=X['Cabin'].str.extract(r'([a-zA-Z])',expand=False).value_counts().index[0]
    return self

  def transform(self,X):

    X=X.copy()
    X['Title']=X['Name'].str.extract(r'\,\s+(\w+)')
    X.loc[X['Title'].isin(['Mlle','Ms']),'Title']='Miss'
    X.loc[X['Title'].isin(['Major','Col','Capt']),'Title']='Army'
    X.loc[X['Title'].isin(['Sir','Don','Jonkheer','Lady','the','Mme','Army','Rev','Dr']),'Title']='Rare'
    X['Cabin']=X['Cabin'].str.extract(r'([a-zA-Z])',expand=False)
    X.loc[X['Cabin'].notna() & ~X['Cabin'].isin(['C', 'E', 'G', 'D', 'A', 'B', 'F']),'Cabin']=self.most_occupied_cabin_
    return X

  #i have decided to pass y in fit
  #most_occupied_cabin_ last underscore to show it is in fit.

In [75]:
columns_to_be_dropped=['PassengerId','Name','Ticket']

In [79]:
def drop_columns(X):
  return X.drop(columns=columns_to_be_dropped)

In [83]:
num_columns=['Age','Fare']

In [86]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer

In [88]:
num_pipeline=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
cat_pipeline=make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore',sparse_output=False))

In [89]:
preprocessing=ColumnTransformer(
    [
        ('num',num_pipeline,num_columns),
        ('cat',cat_pipeline,make_column_selector(dtype_include=object))
    ],
    verbose_feature_names_out=False
)

In [127]:
from sklearn.model_selection import train_test_split

In [90]:
X=og_train.drop(columns=['Survived'])
y=og_train['Survived']

In [128]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y)

In [158]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import FunctionTransformer

In [159]:
model_pipeline=Pipeline([
    ('feature_engineer',TitanicDP()),
    ('feature_selection',FunctionTransformer(lambda X:X.drop(columns=columns_to_be_dropped))),
    ('pipeline',preprocessing),
    ('model',RandomForestClassifier(random_state=42))
])

In [160]:
model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('feature_engineer', TitanicDP()),
                ('feature_selection',
                 FunctionTransformer(func=<function <lambda> at 0x7a82ae602700>)),
                ('pipeline',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7a82dbfea510>)],
                                   verbose_feature_names_out=False)),
                ('model', RandomForestClassifier(random_state=42))])

In [161]:
y_pred=(model_pipeline.predict(X_test)[:]>0.5).astype(int)

In [162]:
y_pred

array([0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       1, 1, 0])

In [163]:
from sklearn.metrics import accuracy_score

In [164]:
accuracy_score(y_test,y_pred)

0.7597765363128491

In [165]:
model_pipeline[-1].get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [153]:
param_grid=[{
    "model__max_features":[1,10,15,8],
    "model__max_depth":[None,4,6],
    "model__n_estimators":[100,150,50]
}]

In [166]:
from sklearn.model_selection import GridSearchCV

In [167]:
grid_search=GridSearchCV(model_pipeline,param_grid,cv=3,scoring='accuracy')

In [168]:
grid_search.fit(X_train,y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('feature_engineer', TitanicDP()),
                                       ('feature_selection',
                                        FunctionTransformer(func=<function <lambda> at 0x7a82ae602700>)),
                                       ('pipeline',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'Fare']),
                                                                        ('cat',
                                                                         Pipeline(st...
                                                                                          OneHotEncoder(handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7a82dbfea510>)],
                                                          verbose_feature_names_out=False)),
                                       ('model',
                                        RandomForestClassifier(random_state=42))]),
             param_grid=[{'model__max_depth': [None, 4, 6],
                          'model__max_features': [1, 10, 15, 8],
                          'model__n_estimators': [100, 150, 50]}],
             scoring='accuracy')

In [171]:
grid_results=pd.DataFrame(grid_search.cv_results_)

In [174]:
grid_results.sort_values(by='mean_test_score',ascending=False,inplace=True)

In [175]:
grid_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__max_depth,param_model__max_features,param_model__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
28,0.219391,0.007881,0.022697,0.000574,6,10,150,"{'model__max_depth': 6, 'model__max_features':...",0.789916,0.814346,0.822785,0.809016,0.013938,1
27,0.144543,0.003357,0.017770,0.001010,6,10,100,"{'model__max_depth': 6, 'model__max_features':...",0.781513,0.814346,0.822785,0.806214,0.017803,2
29,0.083137,0.006157,0.015168,0.000186,6,10,50,"{'model__max_depth': 6, 'model__max_features':...",0.789916,0.814346,0.814346,0.806203,0.011516,3
6,0.176130,0.004548,0.018871,0.000504,None,15,100,"{'model__max_depth': None, 'model__max_feature...",0.794118,0.789030,0.835443,0.806197,0.020784,4
8,0.097051,0.004655,0.015726,0.000085,None,15,50,"{'model__max_depth': None, 'model__max_feature...",0.794118,0.784810,0.839662,0.806197,0.023967,4
15,0.180193,0.032897,0.020075,0.003760,4,10,100,"{'model__max_depth': 4, 'model__max_features':...",0.773109,0.801688,0.835443,0.803413,0.025477,6
35,0.157156,0.036482,0.025205,0.007724,6,8,50,"{'model__max_depth': 6, 'model__max_features':...",0.781513,0.805907,0.822785,0.803402,0.016942,7
31,0.283251,0.078907,0.025764,0.006761,6,15,150,"{'model__max_depth': 6, 'model__max_features':...",0.781513,0.805907,0.822785,0.803402,0.016942,7
33,0.243638,0.017343,0.031041,0.002207,6,8,100,"{'model__max_depth': 6, 'model__max_features':...",0.781513,0.805907,0.822785,0.803402,0.016942,7
16,0.199339,0.006510,0.021277,0.000800,4,10,150,"{'model__max_depth': 4, 'model__max_features':...",0.768908,0.810127,0.827004,0.802013,0.024402,10


In [176]:
grid_search.best_params_

{'model__max_depth': 6, 'model__max_features': 10, 'model__n_estimators': 150}

In [177]:
grid_search.best_score_

np.float64(0.809015589358106)

In [179]:
best_model=grid_search.best_estimator_

In [182]:
best_model['model'].feature_importances_

array([0.12977091, 0.18304347, 0.23449483, 0.18608386, 0.00226614,
       0.00327665, 0.04970622, 0.00555696, 0.01696914, 0.00173017,
       0.00215007, 0.01345794, 0.0076222 , 0.01315609, 0.01036382,
       0.01128523, 0.10512227, 0.01933523, 0.00460881])

In [192]:
pd.DataFrame(sorted(zip(best_model['pipeline'].get_feature_names_out(),best_model['model'].feature_importances_)))

,0,1
0,Age,0.129771
1,Cabin_A,0.002266
2,Cabin_B,0.003277
3,Cabin_C,0.049706
4,Cabin_D,0.005557
5,Cabin_E,0.016969
6,Cabin_F,0.001730
7,Cabin_G,0.002150
8,Embarked_C,0.013458
9,Embarked_Q,0.007622


In [193]:
y_pred=best_model.predict(og_test)

In [196]:
submission=pd.DataFrame({
    'PassengerID':og_test['PassengerId'],
    "Survived": best_model.predict(og_test)
})

In [198]:
submission.to_csv('submission.csv',index=False)

In [199]:
from google.colab import files

In [200]:
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>